In [11]:
from pathlib import Path
import pandas as pd

ROOT = Path("..")
BCN_CSV = ROOT / 'data' / 'BCN20000' / 'bcn20000.csv'
HAM_CSV = ROOT / 'data' / 'HAM10000' / 'HAM10000.csv'

BCN_OUT = ROOT / 'data' / 'BCN20000' / 'bcn_test_for_cam.csv'
HAM_OUT = ROOT / 'data' / 'HAM10000' / 'ham_test_for_cam.csv'

print('BCN input:', BCN_CSV)
print('HAM input:', HAM_CSV)
print('BCN output:', BCN_OUT)
print('HAM output:', HAM_OUT)

BCN input: ../data/BCN20000/bcn20000.csv
HAM input: ../data/HAM10000/HAM10000.csv
BCN output: ../data/BCN20000/bcn_test_for_cam.csv
HAM output: ../data/HAM10000/ham_test_for_cam.csv


In [12]:
PRIMARY_MAP = {
    'melanoma nos': 'MEL',
    'nevus': 'NV',
    'basal cell carcinoma': 'BCC',
    'solar or actinic keratosis': 'AKIEC',
    'pigmented benign keratosis': 'BKL',
    'seborrheic keratosis': 'BKL',
    'solar lentigo': 'BKL',
    'dermatofibroma': 'DF',
}

PRIMARY_EXCLUDE = {
    'squamous cell carcinoma': 'exclude_scc',
    'squamous cell carcinoma nos': 'exclude_scc',
    'scc': 'exclude_scc',
    'melanoma metastasis': 'exclude_melanoma_metastasis',
    'scar': 'exclude_scar',
    'unknown': 'exclude_unknown',
    'other': 'exclude_other',
}

HAM_DX_TO_UPPER = {
    'akiec': 'AKIEC',
    'bcc': 'BCC',
    'bkl': 'BKL',
    'df': 'DF',
    'mel': 'MEL',
    'nv': 'NV',
    'vasc': 'VASC',
}

In [13]:
def normalize_image_id(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r'\.(jpg|jpeg|png)$', '', regex=True, case=False)
    )


def normalize_text(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower()


def require_columns(df: pd.DataFrame, required: list[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{name} is missing required columns: {missing}. Found columns: {df.columns.tolist()}')

In [14]:
bcn_df = pd.read_csv(BCN_CSV)
require_columns(bcn_df, ['split', 'isic_id', 'diagnosis'], 'BCN20000')

bcn_test = bcn_df.loc[normalize_text(bcn_df['split']) == 'test'].copy()
bcn_test['image'] = normalize_image_id(bcn_test['isic_id'])
bcn_test['diagnosis_norm'] = normalize_text(bcn_test['diagnosis'])
bcn_test['exclude_reason'] = bcn_test['diagnosis_norm'].map(PRIMARY_EXCLUDE)
bcn_test['gt_label'] = bcn_test['diagnosis_norm'].map(PRIMARY_MAP)

excluded_bcn = bcn_test[bcn_test['exclude_reason'].notna()].copy()
unmapped_bcn = bcn_test[bcn_test['gt_label'].isna() & bcn_test['exclude_reason'].isna()].copy()
bcn_test_final = bcn_test[bcn_test['gt_label'].notna()].copy()

first_cols = ['image', 'gt_label']
other_cols = [c for c in bcn_test_final.columns if c not in first_cols]
bcn_test_final = bcn_test_final[first_cols + other_cols]

bcn_test_final.to_csv(BCN_OUT, index=False)

print(f'BCN kept rows: {len(bcn_test_final)}')
print(f'BCN excluded rows: {len(excluded_bcn)}')
print(f'BCN unmapped rows: {len(unmapped_bcn)}')

if len(excluded_bcn):
    print('\nExcluded diagnosis counts:')
    print(excluded_bcn['diagnosis_norm'].value_counts())

if len(unmapped_bcn):
    print('\nUnmapped diagnosis counts:')
    print(unmapped_bcn['diagnosis_norm'].value_counts())

display(bcn_test_final.head())

BCN kept rows: 843
BCN excluded rows: 34
BCN unmapped rows: 365

Excluded diagnosis counts:
diagnosis_norm
squamous cell carcinoma    34
Name: count, dtype: int64

Unmapped diagnosis counts:
diagnosis_norm
melanoma             301
actinic keratosis     55
vascular lesion        9
Name: count, dtype: int64


,image,gt_label,isic_id,attribution,copyright_license,age_approx,anatom_site_general,benign_malignant,diagnosis,diagnosis_confirm_type,image_type,lesion_id,melanocytic,sex,binary_label,split,label,diagnosis_norm,exclude_reason
3,ISIC_0053459,NV,ISIC_0053459,Hospital Clínic de Barcelona,CC-BY-NC,60.0,anterior torso,benign,nevus,histopathology,dermoscopic,IL_7455346,True,male,0,test,2,nevus,NaN
51,ISIC_0053535,BCC,ISIC_0053535,Hospital Clínic de Barcelona,CC-BY-NC,80.0,head/neck,NaN,basal cell carcinoma,NaN,dermoscopic,IL_7720571,False,female,0,test,3,basal cell carcinoma,NaN
55,ISIC_0053540,NV,ISIC_0053540,Hospital Clínic de Barcelona,CC-BY-NC,40.0,lower extremity,benign,nevus,histopathology,dermoscopic,IL_4578013,True,female,0,test,2,nevus,NaN
76,ISIC_0053568,BCC,ISIC_0053568,Hospital Clínic de Barcelona,CC-BY-NC,45.0,anterior torso,NaN,basal cell carcinoma,histopathology,dermoscopic,IL_7136715,False,male,0,test,3,basal cell carcinoma,NaN
77,ISIC_0053569,BKL,ISIC_0053569,Hospital Clínic de Barcelona,CC-BY-NC,50.0,anterior torso,NaN,seborrheic keratosis,histopathology,dermoscopic,IL_6301457,False,male,0,test,4,seborrheic keratosis,NaN


In [15]:
ham_df = pd.read_csv(HAM_CSV)
require_columns(ham_df, ['split', 'image_id', 'dx'], 'HAM10000')

ham_test = ham_df.loc[normalize_text(ham_df['split']) == 'test'].copy()
ham_test['image'] = normalize_image_id(ham_test['image_id'])
ham_test['dx_norm'] = normalize_text(ham_test['dx'])
ham_test['gt_label'] = ham_test['dx_norm'].map(HAM_DX_TO_UPPER)

unmapped_ham = ham_test[ham_test['gt_label'].isna()].copy()
ham_test_final = ham_test[ham_test['gt_label'].notna()].copy()

first_cols = ['image', 'gt_label']
other_cols = [c for c in ham_test_final.columns if c not in first_cols]
ham_test_final = ham_test_final[first_cols + other_cols]

ham_test_final.to_csv(HAM_OUT, index=False)

print(f'HAM kept rows: {len(ham_test_final)}')
print(f'HAM unmapped rows: {len(unmapped_ham)}')

if len(unmapped_ham):
    print('\nUnmapped dx counts:')
    print(unmapped_ham['dx_norm'].value_counts())

display(ham_test_final.head())

HAM kept rows: 1232
HAM unmapped rows: 0


,image,gt_label,lesion_id,image_id,dx,dx_type,age,sex,localization,dataset,split,label,binary_label,age_group,dx_norm
8783,ISIC_0032343,BKL,HAM_0006071,ISIC_0032343,bkl,histo,70.0,female,face,vidir_modern,test,2,0,old,bkl
8784,ISIC_0024698,NV,HAM_0001751,ISIC_0024698,nv,consensus,70.0,male,face,vidir_modern,test,5,0,old,nv
8785,ISIC_0031624,BKL,HAM_0002092,ISIC_0031624,bkl,histo,35.0,female,lower extremity,vidir_modern,test,2,0,medium,bkl
8786,ISIC_0025510,BKL,HAM_0002092,ISIC_0025510,bkl,histo,35.0,female,lower extremity,vidir_modern,test,2,0,medium,bkl
8787,ISIC_0025388,BKL,HAM_0003007,ISIC_0025388,bkl,histo,40.0,female,abdomen,vidir_modern,test,2,0,medium,bkl


In [16]:
for path in [BCN_OUT, HAM_OUT]:
    df = pd.read_csv(path)
    assert 'image' in df.columns, f'{path.name} is missing image column'
    assert 'gt_label' in df.columns, f'{path.name} is missing gt_label column'
    assert not df['image'].astype(str).str.contains(r'\.(jpg|jpeg|png)$', case=False, regex=True).any(), (
        f'{path.name} still contains file extensions in image column'
    )
    print(f'OK: {path.name}')
    print(df['gt_label'].value_counts().sort_index())
    print()

OK: bcn_test_for_cam.csv
gt_label
BCC    284
BKL    117
DF      13
NV     429
Name: count, dtype: int64

OK: ham_test_for_cam.csv
gt_label
AKIEC     35
BCC       44
BKL      107
DF         8
MEL       70
NV       951
VASC      17
Name: count, dtype: int64



/var/folders/8q/87zqgq3546j682gq4dl7t0b00000gn/T/ipykernel_66488/869658240.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  assert not df['image'].astype(str).str.contains(r'\.(jpg|jpeg|png)$', case=False, regex=True).any(), (
/var/folders/8q/87zqgq3546j682gq4dl7t0b00000gn/T/ipykernel_66488/869658240.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  assert not df['image'].astype(str).str.contains(r'\.(jpg|jpeg|png)$', case=False, regex=True).any(), (
